# 01 — Data exploration (Phase 1)

**Goal:** characterize the arsenic well-test export and the Kent County mosquito surveillance workbook *before* dashboard wiring.

**Critical:** `Result_Cat` thresholds must be confirmed with the data owner (see arsenic section). The notebook documents **data-derived** rules that match every labeled row in the current file, but those are not a substitute for documented lab or program policy.

---

**Kernel:** Select the project virtualenv (`…/environmental-health-dashboard/venv`) as the notebook kernel, and run `pip install -r requirements.txt` so `nbformat`, `ipython`, and `ipykernel` are available if you use Plotly elsewhere.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd

def project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise RuntimeError("Run this notebook from the repo (needs data/raw).")

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw"
ARSENIC_XLSX = RAW / "ALL_ArsenicTests_Mapping_071524.xlsx"
MOSQUITO_XLSX = RAW / "mosquitoSurveillanceData_2021_Kent (Final).xlsx"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## Arsenic dataset — inventory


In [ ]:
arsenic = pd.read_excel(ARSENIC_XLSX, sheet_name="Sheet1")

print("shape:", arsenic.shape)
print("\ndtypes:\n", arsenic.dtypes)
print("\nmissing per column:\n", arsenic.isna().sum())
print("\ncolumn names:", list(arsenic.columns))


In [ ]:
print(arsenic.describe(include="all").T.head(20))

num_cols = arsenic.select_dtypes(include=[np.number]).columns
cat_cols = [c for c in arsenic.columns if c not in num_cols]

print("\n--- Categorical uniques (truncated) ---")
for c in cat_cols:
    u = arsenic[c].dropna().unique()
    print(f"\n{c}: n_unique={len(u)}")
    print(u[:25])

print("\n--- Numeric summary ---")
print(arsenic[num_cols].describe())

print("\nDuplicate rows (all columns):", int(arsenic.duplicated().sum()))


In [ ]:
# The source column is named Year but stores full dates (sample test dates).
year_series = pd.to_datetime(arsenic["Year"], errors="coerce")
print("test date range:", year_series.min(), "→", year_series.max())
print("calendar years present:", sorted(year_series.dt.year.dropna().unique().tolist()))


### Arsenic — geography and units

- **Coordinates:** not present in this export; mapping will rely on **addresses** (`Street`, `City`, `State`, `Zip`, `Full_Add`, `Prop_Add`) unless lat/lon are joined later.
- **Units:** `Result` is numeric on the order of **mg/L** (values align with common arsenic drinking-water reporting; treat as *assumed* until confirmed on the lab report metadata).
- **`Result_Cat`:** stored as numeric codes `0.0`–`3.0` with 31 missing values where `Result` is still populated (likely entry gaps).

**Action for your team:** ask what official bins define `Result_Cat` (lab MDL vs policy bands vs MCL comparisons). Until then, the cleaning script uses thresholds inferred below.


In [ ]:
labeled = arsenic.dropna(subset=["Result_Cat"])
for code in sorted(labeled["Result_Cat"].unique()):
    r = labeled.loc[labeled["Result_Cat"] == code, "Result"]
    print(f"Result_Cat {int(code)}: n={len(r)} min={r.min():.5g} max={r.max():.5g}")

# Data-derived breakpoints (validated: zero mismatches vs labeled rows)

def infer_cat(v: float):
    if pd.isna(v):
        return np.nan
    if v <= 0:
        return 0.0
    if v < 0.005:
        return 1.0
    if v < 0.01:
        return 2.0
    return 3.0

labeled = labeled.assign(_inf=labeled["Result"].map(infer_cat))
print("\nmismatch count vs inferred:", int((labeled["Result_Cat"] != labeled["_inf"]).sum()))
print("Inferred policy text: 0=0 mg/L; 1=(0,0.005); 2=[0.005,0.01); 3=>=0.01 mg/L (ties to EPA 0.01 mg/L MCL reference).")


## Mosquito surveillance — inventory


In [ ]:
mosquito = pd.read_excel(MOSQUITO_XLSX, sheet_name="mosquitoSurveillanceData")

print("shape:", mosquito.shape)
print("\nmissing (top):\n", mosquito.isna().sum().sort_values(ascending=False).head(12))
print("\nDuplicate rows:", int(mosquito.duplicated().sum()))

set_dt = pd.to_datetime(mosquito["Trap Set Date"], errors="coerce")
pick_dt = pd.to_datetime(mosquito["Trap Pickup Date"], errors="coerce")
print("\nTrap set range:", set_dt.min(), "→", set_dt.max())
print("Trap pickup range:", pick_dt.min(), "→", pick_dt.max())


In [ ]:
for col in ["Genus", "Species", "Type of Collection Site", "Collection Method/Trap Type", "Detected"]:
    print(f"\n== {col} ==")
    print(mosquito[col].value_counts(dropna=False).head(15))

geo_cols = [c for c in mosquito.columns if any(k in c.lower() for k in ["address", "city", "county", "state", "lat", "lon", "coord"])]
print("\nGeographic / location columns:", geo_cols)


In [ ]:
# Detection frequency among rows with a known outcome
mask = mosquito["Detected"].notna()
print("Rows with Detected not null:", int(mask.sum()))
print(mosquito.loc[mask, "Detected"].value_counts())

# Species distribution (raw Species column includes many literal "sp.")
print("\nTop species strings:")
print(mosquito["Species"].value_counts().head(10))


## Phase 1 takeaways

| Dataset | Rows | Key geo fields | Key time field | Caveats |
|---------|------|----------------|------------------|---------|
| Arsenic | 1,507 | Mailing-style addresses, ZIP float | `Year` holds **dates** | No lat/lon; confirm mg/L + `Result_Cat` policy |
| Mosquito | 528 | Street/city/county/state | Trap set/pickup dates | 173 rows lack genus/species/detected together (empty or not ID'd) |

Run the cleaning modules from the repo root:

`PYTHONPATH=. python -m src.cleaning.clean_arsenic` and `PYTHONPATH=. python -m src.cleaning.clean_mosquito`
